# 🍷 Wine Quality Prediction - Progetto Capstone

## Contesto del Progetto

### Il Cliente: Cantina di Pregio Portoghese

La nostra cantina cliente, produttrice di **Vinho Verde** (vini verdi portoghesi), si trova di fronte a una sfida cruciale nel processo di produzione: **quale lotti di vino destinare all'affinamento in barrique** per ottenere vini di alta qualità destinati al mercato premium.

### La Richiesta dell'Enologo

L'enologo capo della cantina ci ha contattato con una richiesta specifica:

> *"Ogni anno produciamo centinaia di lotti di vino. Attualmente, la decisione su quali lotti far invecchiare nelle costose botti di rovere si basa principalmente sull'esperienza e sull'assaggio. Questo processo è:*
> - *Soggettivo e dipendente dalla disponibilità dell'enologo*
> - *Costoso in termini di tempo (richiede settimane per le valutazioni)*
> - *Rischioso: investiamo in barrique costose senza certezza del risultato finale*
>
> *Abbiamo bisogno di un sistema che, analizzando i parametri chimico-fisici del vino in fase iniziale, predica con affidabilità se un lotto merita l'investimento per l'affinamento premium."*

### Raccolta Dati

La cantina ha messo a disposizione il proprio **database storico** contenente:
- **Analisi chimico-fisiche** di oltre 6,000 campioni di vino
- **Valutazioni sensoriali** da parte di panel di esperti sommelier (score 0-10)
- Parametri misurati in laboratorio: acidità, pH, solfiti, alcol, residui zuccherini, ecc.

I dati provengono da:
1. **Laboratorio interno** della cantina (misure oggettive)
2. **Panel di degustazione certificato** (valutazione qualità)
3. **Archivio storico** delle decisioni di affinamento e risultati commerciali

### Dataset: UCI Wine Quality

Per questo progetto utilizziamo il dataset pubblico **UCI Wine Quality** (ID: 186), che rappresenta esattamente lo scenario descritto:
- **6,497 campioni** di vini Vinho Verde (rossi e bianchi)
- **11 variabili chimico-fisiche** misurate in laboratorio
- **Valutazione qualitativa** (0-10) da esperti enologi

### Obiettivo del Progetto

Sviluppare un **sistema di Machine Learning** che:
1. Predica la **probabilità che un vino sia di alta qualità** (score ≥ 7)
2. Supporta l'enologo nelle **decisioni di affinamento** 

### Deliverable

1. **Modello ML** addestrato e validato (pipeline scikit-learn)
2. **Webapp Streamlit** per l'utilizzo in cantina
3. **Sistema MLflow** per tracking e versionamento
4. **Explainability** (SHAP, Feature Importance) per trasparenza decisionale

---

## Esplorazione Dataset e Sviluppo Modello

In [1]:
from ucimlrepo import fetch_ucirepo 
import os
import mlflow

# Opt-in necessario nelle versioni recenti per usare il backend file legacy.
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
mlflow.set_tracking_uri("file:mlruns")  


# fetch dataset 
wine_quality = fetch_ucirepo(id=186) 
  
# data (as pandas dataframes) 
X = wine_quality.data.features 
y = wine_quality.data.targets 
  
# metadata 
print(wine_quality.metadata) 
  
# variable information 
print(wine_quality.variables) 


/Users/andreacorvaglia/Library/Mobile Documents/com~apple~CloudDocs/Corsi/playground/ai-course-test-deploy/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'uci_id': 186, 'name': 'Wine Quality', 'repository_url': 'https://archive.ics.uci.edu/dataset/186/wine+quality', 'data_url': 'https://archive.ics.uci.edu/static/public/186/data.csv', 'abstract': 'Two datasets are included, related to red and white vinho verde wine samples, from the north of Portugal. The goal is to model wine quality based on physicochemical tests (see [Cortez et al., 2009], http://www3.dsi.uminho.pt/pcortez/wine/).', 'area': 'Business', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 4898, 'num_features': 11, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['quality'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Wed Nov 15 2023', 'dataset_doi': '10.24432/C56S3T', 'creators': ['Paulo Cortez', 'A. Cerdeira', 'F. Almeida', 'T. Matos', 'J. Reis'], 'intro_paper': {'ID': 252, 'type': 'NATIVE', 'title': 'Modeling wine preferences

In [2]:
# Binarizza la variabile y: 1 se quality >= 7, altrimenti 0
y_bin = (y['quality'] >= 7).astype(int)
# rename in high_quality
y_bin.rename('high_quality', inplace=True)
print(y_bin.value_counts())

high_quality
0    5220
1    1277
Name: count, dtype: int64


---

## 📋 Guida al Progetto

### Requisiti Minimi 

Per completare il progetto, è **necessario** eseguire i seguenti passi:

1. ✅ **Allenare un modello di Machine Learning**
   - Utilizzare i dati del wine quality dataset
   - Creare una pipeline scikit-learn completa (preprocessing + model)
   - Valutare le performance del modello

2. ✅ **Loggare il modello su MLflow**
   - Configurare MLflow tracking
   - Loggare parametri, metriche e artefatti del modello
   - Salvare la pipeline completa con `mlflow.sklearn.log_model(..., serialization_format="cloudpickle")` (come in T4)

   💡 La sidebar della webapp mostra le metriche **solo** se sono loggate con questi nomi esatti: `accuracy`, `precision`, `recall`, `f1_score`, `roc_auc`

3. ✅ **Registrare il modello nel Model Registry**
   - Registrare il modello con nome `wine_clf`
   - Assegnare l'alias `@production` alla versione migliore
   - Verificare che il modello sia accessibile via registry

4. ✅ **Testare la Webapp**
   - Eseguire `uv run streamlit run app.py` dalla cartella `capstone_project`
   - Verificare che carichi correttamente il modello
   - Testare le predizioni con diversi parametri

---

### Passi Avanzati (Opzionali ma Consigliati)

Una volta completati i requisiti minimi, potete approfondire con:

#### **1. Valutazione Avanzata delle Performance**

**Metriche sul Test Set:**
- Calcolare **ROC-AUC** per valutare la capacità discriminativa
- Analizzare **Precision-Recall Curve** per trovare il trade-off ottimale
- Confrontare diverse soglie di classificazione

**Domanda Critica da Rispondere:**
> Nel contesto della cantina, **quale errore è più costoso?**
> - **Falso Positivo**: Affinare in barrique un vino che poi risulta mediocre → spreco investimento
> - **Falso Negativo**: Non affinare un vino eccellente → perdita opportunità di vendita premium
> 
> In base alla risposta, ottimizzare per **Precision** (minimizzare FP) o **Recall** (minimizzare FN)

#### **2. Model Comparison e Selection**

- **Testare diversi algoritmi**:
  - Random Forest (baseline)
  - Logistic Regression (interpretabilità)
  - SVM con kernel RBF
  
- **Loggare tutti i modelli su MLflow** per confronto sistematico
- **Selezionare il migliore** in base a metrica rilevante + interpretabilità

#### **3. Explainability e Feature Importance**

**Permutation Feature Importance (PFI):**
- Identificare quali feature contribuiscono maggiormente alle predizioni
- Loggare plot PFI come artefatto MLflow

**SHAP Values:**
- Calcolare SHAP values per interpretabilità locale
- Generare summary plot e dependence plot
- Loggare visualizzazioni su MLflow

**Domande da Esplorare:**
- Quali parametri chimico-fisici sono più importanti per la qualità?
- Ci sono interazioni tra feature? (es. alcol + acidità)
- Il modello è allineato con la conoscenza enologica?

#### **4. Iterazione e Miglioramento**

1. **Esplorare le diverse run MLflow**:
   - Usare MLflow UI: `MLFLOW_ALLOW_FILE_STORE=true uv run mlflow ui --backend-store-uri file:mlruns --port 5000` (dalla cartella `capstone_project`)
   - Confrontare run parallele
   - Identificare pattern nei parametri vincenti

2. **Aggiornare la webapp**:
   - Registrare il modello migliorato con alias `@production`
   - Riavviare la webapp
   - Verificare che le predizioni siano migliorate

3. **Documentare il processo**:
   - Motivare la scelta del modello finale
   - Spiegare il trade-off precision/recall scelto
   - Documentare le feature più importanti

---

**Iniziate con i requisiti minimi, poi espandete progressivamente con i passi avanzati!**

In [3]:
# ==================== 1. TRAINING DEL MODELLO ====================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

# Train/test split stratificato: la classe "alta qualità" è solo ~20% dei campioni
X_train, X_test, y_train, y_test = train_test_split(
    X, y_bin,
    test_size=0.2,
    random_state=42,
    stratify=y_bin
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Quota alta qualità - train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

Train: (5197, 11), Test: (1300, 11)
Quota alta qualità - train: 0.196, test: 0.197


In [4]:
# Pipeline completa: preprocessing + modello.
# Tutte le feature sono numeriche, quindi basta uno scaler (non serve al Random Forest,
# ma rende la pipeline riutilizzabile anche con modelli lineari o SVM).
# Niente class_weight="balanced": la webapp usa soglie (15/30/50%) pensate per
# probabilità vicine a quelle reali, con una base del ~20% di vini di alta qualità.
params = {
    "n_estimators": 300,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "random_state": 42,
}

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_jobs=-1, **params)),
])
pipeline.fit(X_train, y_train)

# Valutazione sul test set
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

# Nomi esatti richiesti dalla sidebar della webapp
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
}
print(pd.Series(metrics).round(4))
print()
print(classification_report(y_test, y_pred, target_names=["standard", "alta qualità"]))

accuracy     0.8885
precision    0.8033
recall       0.5742
f1_score     0.6697
roc_auc      0.9148
dtype: float64

              precision    recall  f1-score   support

    standard       0.90      0.97      0.93      1044
alta qualità       0.80      0.57      0.67       256

    accuracy                           0.89      1300
   macro avg       0.85      0.77      0.80      1300
weighted avg       0.88      0.89      0.88      1300



In [5]:
# ==================== 2. LOG DEL MODELLO SU MLFLOW ====================
from mlflow.models import infer_signature

mlflow.set_experiment("wine_quality_capstone")

with mlflow.start_run(run_name="RandomForest") as run:
    # Parametri
    mlflow.log_params(params)
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("test_size", 0.2)

    # Metriche sul test set
    mlflow.log_metrics(metrics)

    # Tag
    mlflow.set_tag("dataset", "uci_wine_quality_186")
    mlflow.set_tag("target", "quality >= 7")

    # Pipeline completa con signature e input example
    signature = infer_signature(X_train, pipeline.predict_proba(X_train)[:, 1])
    model_info = mlflow.sklearn.log_model(
        pipeline,
        name="model",
        signature=signature,
        input_example=X_train.head(3),
        serialization_format="cloudpickle",
    )

run_id = run.info.run_id
print(f"Run ID: {run_id}")
print(f"Model URI: {model_info.model_uri}")

2026/10/08 08:53:46 INFO mlflow.tracking.fluent: Experiment with name 'wine_quality_capstone' does not exist. Creating a new experiment.


2026/10/08 08:53:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID: a8e77ea133434cf0b4f23d977a242679
Model URI: models:/m-9f178075a77242ad8d64d8a89668e89a


In [6]:
# ==================== 3. MODEL REGISTRY ====================
from mlflow.tracking import MlflowClient

model_name = "wine_clf"
client = MlflowClient()

# Registra il modello loggato e assegna l'alias @production
model_version = mlflow.register_model(model_info.model_uri, model_name)
client.set_registered_model_alias(model_name, "production", model_version.version)
print(f"Registrato {model_name} versione {model_version.version} con alias @production")

# Verifica: il modello è accessibile via registry e dà le stesse predizioni
loaded = mlflow.sklearn.load_model(f"models:/{model_name}@production")
loaded_proba = loaded.predict_proba(X_test)[:, 1]
assert (loaded_proba == y_proba).all()
print(f"Caricato da models:/{model_name}@production - ROC-AUC sul test set: {roc_auc_score(y_test, loaded_proba):.4f}")
print(f"Feature attese dal modello: {list(loaded.feature_names_in_)}")

Registrato wine_clf versione 1 con alias @production
Caricato da models:/wine_clf@production - ROC-AUC sul test set: 0.9148
Feature attese dal modello: ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol']


Successfully registered model 'wine_clf'.
Created version '1' of model 'wine_clf'.


### 4. Test della Webapp

Dalla cartella `capstone_project`:

```bash
uv run streamlit run app.py
```

La webapp carica `wine_clf@production` dal registry, mostra nella sidebar le metriche loggate
(`accuracy`, `precision`, `recall`, `f1_score`, `roc_auc`) e aggiorna la predizione a ogni modifica degli slider.
Provare i lotti di esempio: "Rosso strutturato" deve avere una probabilità molto più alta di "Rosso difettoso".

### Nota Importante

Il codice riportato sotto deve essere eseguito **alla fine** del workflow. Questo passaggio è necessario per salvare il modello migliore in una cartella locale, che potrà essere utilizzata per il **deploy** su **Streamlit Cloud**. 

Assicurati che il modello sia stato registrato e che la versione in produzione sia stata correttamente selezionata prima di eseguire il codice.

In [7]:
import shutil
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

# Inizializza client MLflow
client = MlflowClient()
# Nome del modello registrato
model_name = "wine_clf"

model_uri = f"models:/{model_name}@production"

# Carica il modello
model = mlflow.sklearn.load_model(model_uri)

# Salva il modello in una cartella locale (model/model.pkl, letto dalla webapp come fallback)
shutil.rmtree("model", ignore_errors=True)  # sovrascrive un eventuale export precedente
mlflow.sklearn.save_model(model, "model", serialization_format="cloudpickle")


2026/10/08 08:53:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
